# TradeFlow AI — nb2_chromadb_index

Generates the HS code ChromaDB vector index using realistic hierarchical BTKI data.
Diperbarui: Menyertakan HS Code spesifik dari Ground Truth untuk keperluan evaluasi.

In [ ]:
!pip install -q chromadb sentence-transformers pandas

In [ ]:
import chromadb
import pandas as pd

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="hs_codes")

# Mock INSW Data: Mengandung Automotive (Bab 87) sebagai pengecoh,
# ditambah data spesifik dari Ground Truth (Bab 84, 28, 72) agar RAG berhasil menemukan jawaban yang benar.
data = [
    # --- KUNCI JAWABAN (Dari TradeFlow_GroundTruth_v5.2.json) ---
    {"hs": "84821000", "desc": "Bantalan peluru. Ball bearings."},
    {"hs": "84822000", "desc": "Bantalan rol tirus, termasuk rakitan kerucut dan rol tirus. Tapered roller bearings, including cone and tapered roller assemblies."},
    {"hs": "84825000", "desc": "Bantalan rol silinder lainnya. Other cylindrical roller bearings."},
    {"hs": "84828000", "desc": "Lain-lain, termasuk bantalan kombinasi peluru atau rol. Other, including combined ball or roller bearings."},
    {"hs": "28151110", "desc": "Natrium hidroksida (soda kaustik) dalam bentuk padat (flakes). Sodium hydroxide (caustic soda) in solid form (flakes)."},
    {"hs": "72193590", "desc": "Produk canai lincin dari baja tahan karat, dengan ketebalan kurang dari 0,5 mm. Flat-rolled products of stainless steel, of a thickness of less than 0.5 mm."},
    
    # --- DATA Pengecoh (Automotive BTKI Chapter 87) ---
    {"hs": "87070000", "desc": "Bodi (termasuk kabin), untuk kendaraan bermotor dari pos 87.01 sampai dengan 87.05. Bodies (including cabs), for the motor vehicles of headings 87.01 to 87.05."},
    {"hs": "87071010", "desc": "Bodi Untuk gokart dan mobil golf (termasuk golf buggy) dan kendaraan semacam itu. For go-karts and golf cars (including golf buggies) and similar vehicles."},
    {"hs": "87071020", "desc": "Bodi Untuk ambulan. For ambulances."},
    {"hs": "87071030", "desc": "Bodi Untuk kendaraan yang dirancang secara khusus untuk perjalanan di atas salju. For vehicles specially designed for travelling on snow."},
    {"hs": "87079011", "desc": "Kabin pengemudi untuk kendaraan dari subpos 8701.21, 8701.22, 8701.23, 8701.24 atau 8701.29. Driver's cabin for vehicles..."},
    {"hs": "87079021", "desc": "Bodi Untuk mobil (termasuk limousin panjang tetapi tidak termasuk coach, bus, minibus atau van). For motor cars (including stretch limousines...)"},
    {"hs": "87080000", "desc": "Bagian dan aksesori kendaraan bermotor dari pos 87.01 sampai dengan 87.05. Parts and accessories of the motor vehicles of headings 87.01 to 87.05."},
    {"hs": "87081010", "desc": "Bumper dan bagiannya Untuk kendaraan dari pos 87.01. Bumpers and parts thereof For vehicles of heading 87.01."},
    {"hs": "87082100", "desc": "Sabuk pengaman. Safety seat belts."},
    {"hs": "87082200", "desc": "Kaca depan (windshield), jendela belakang dan jendela lainnya yang dirinci pada Catatan Subpos 1 Bab ini. Front windscreens (windshields), rear windows..."}
]

df = pd.DataFrame(data)

print(f"Indexing {len(df)} HS codes into ChromaDB...")
collection.add(
    documents=df['desc'].tolist(),
    metadatas=[{"hs_code": hs} for hs in df['hs']],
    ids=df['hs'].tolist()
)
print("Indexing complete. Database saved to ./chroma_db")
print("Ready to be downloaded and used by the TradeFlow API.")
